# 🌾 DRAFT 03 — Essais de Modélisation

**Auteure :** Souha Guezguez  
**Date :** Mars 2026  
**Statut :** Brouillon — essais et comparaisons de modèles

> Ce notebook documente mes essais de modélisation avant d'arriver au modèle final.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Chargement et preprocessing rapide
df = pd.read_csv('../data/dataset_secheresse_tunisie.csv', parse_dates=['date'])
df = df.dropna(subset=['risque_secheresse'])
df['risque_secheresse'] = df['risque_secheresse'].astype(int)
df['mois'] = df['date'].dt.month
df = df.sort_values(['gouvernorat','date'])

le_reg  = LabelEncoder()
le_gouv = LabelEncoder()
df['region_enc'] = le_reg.fit_transform(df['region'])
df['gouvernorat_enc'] = le_gouv.fit_transform(df['gouvernorat'])
df['precip_rolling_30j'] = df.groupby('gouvernorat')['precipitations_mm'].transform(lambda x: x.rolling(30, min_periods=1).mean())
df['precip_rolling_7j']  = df.groupby('gouvernorat')['precipitations_mm'].transform(lambda x: x.rolling(7,  min_periods=1).mean())
df['amplitude_thermique']= df['temperature_max'] - df['temperature_min']
df['jour_sec']           = (df['precipitations_mm'] == 0).astype(int)

def get_saison(m):
    if m in [12,1,2]: return 1
    elif m in [3,4,5]: return 2
    elif m in [6,7,8]: return 3
    else: return 4
df['saison'] = df['mois'].apply(get_saison)

FEATURES = ['precipitations_mm','temperature_moy','temperature_max','temperature_min',
            'humidite_pct','vent_ms','precip_rolling_7j','precip_rolling_30j',
            'amplitude_thermique','jour_sec','saison','mois','region_enc','gouvernorat_enc']

X = df[FEATURES]
y = df['risque_secheresse']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## Essai 1 — Logistic Regression (baseline)

In [ ]:
# Essai avec différents solvers
for solver in ['lbfgs', 'saga']:
    lr = LogisticRegression(max_iter=1000, class_weight='balanced',
                            solver=solver, random_state=42)
    lr.fit(X_train_sc, y_train)
    f1 = f1_score(y_test, lr.predict(X_test_sc), average='weighted')
    print(f'LR solver={solver}: F1={f1:.4f}')
print('\n→ CONCLUSION: LR trop faible (60%) - problème non-linéaire')

## Essai 2 — Decision Tree avec différentes profondeurs

In [ ]:
# Impact de max_depth
print('Decision Tree - impact de max_depth:')
for depth in [5, 10, 15, 20, None]:
    dt = DecisionTreeClassifier(max_depth=depth, class_weight='balanced', random_state=42)
    dt.fit(X_train_sc, y_train)
    f1_train = f1_score(y_train, dt.predict(X_train_sc), average='weighted')
    f1_test  = f1_score(y_test,  dt.predict(X_test_sc),  average='weighted')
    print(f'  depth={str(depth):4s}: Train F1={f1_train:.3f} | Test F1={f1_test:.3f} | '
          f'Overfitting={f1_train-f1_test:.3f}')
print('\n→ DÉCISION: max_depth=10 bon compromis biais/variance')

## Essai 3 — Random Forest avec différents n_estimators

In [ ]:
print('Random Forest - impact de n_estimators:')
for n in [10, 50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=n, class_weight='balanced',
                                random_state=42, n_jobs=-1)
    rf.fit(X_train_sc, y_train)
    f1 = f1_score(y_test, rf.predict(X_test_sc), average='weighted')
    print(f'  n_estimators={n:3d}: Test F1={f1:.4f}')
print('\n→ DÉCISION: 100 arbres = bon compromis temps/performance')

## Essai 4 — Cross-validation pour valider la robustesse

In [ ]:
# Cross-validation 5 folds pour vérifier que le modèle est stable
print('Cross-validation (5 folds) — Random Forest:')
rf_cv = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                random_state=42, n_jobs=-1)
scores = cross_val_score(rf_cv, X_train_sc, y_train, cv=5, scoring='f1_weighted')
print(f'  F1 par fold : {[f"{s:.3f}" for s in scores]}')
print(f'  F1 moyen    : {scores.mean():.3f} ± {scores.std():.3f}')
print('\n→ Faible écart-type = modèle stable et robuste ✅')

## Essai 5 — XGBoost vs Random Forest

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost':       XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                   eval_metric='mlogloss', random_state=42, n_jobs=-1)
}
print('Comparaison RF vs XGBoost:')
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    f1  = f1_score(y_test,  model.predict(X_test_sc),  average='weighted')
    acc = accuracy_score(y_test, model.predict(X_test_sc))
    print(f'  {name}: Accuracy={acc:.3f} | F1={f1:.3f}')

print('\n→ Random Forest légèrement supérieur sur ce dataset → retenu comme modèle principal')

## Résumé des conclusions de modélisation

| Modèle | F1 Test | Remarque |
|--------|---------|----------|
| Logistic Regression | ~0.65 | Trop simple, problème non-linéaire |
| Decision Tree | ~0.73 | Overfitting si profondeur trop élevée |
| Random Forest (100) | ~0.83 | ✅ Bon compromis |
| XGBoost | ~0.78 | Besoin de plus d'optimisation |

**Décision finale :** Random Forest optimisé avec RandomizedSearchCV → F1 = 0.873